In [ ]:
# | default_exp article

# Article
> Database model and CRUD operations for tracking article files and their SEO metadata.

In [ ]:
# | export
from sqlmodel import SQLModel, Field, create_engine, Session, select, Column
from sqlalchemy import JSON
from datetime import datetime, date
from seo_rat.sqlite_db import  get_session

In [ ]:
# | exporti
class Article(SQLModel, table=True):
    "An article file tracked for SEO optimization."
    __table_args__ = {"extend_existing": True}
    id: int | None = Field(default=None, primary_key=True)
    website_id: int = Field(foreign_key="website.id") # Parent website
    file_path: str                                     # Path to .md or .ipynb file
    url: str | None = None                             # Public URL of the article
    focus_keyword: str | None = None                   # Primary SEO keyword
    secondary_keywords: list[str] | None = Field(default=None, sa_column=Column(JSON))
    target_goal: str | None = None                     # Optimization goal
    last_optimized: datetime | None = None             # Last optimization timestamp
    created_at: datetime = Field(default_factory=datetime.now)

In [ ]:
# | export
def insert_article(session:Session,   # Active database session
                   article:Article    # Article instance to insert
                  ) -> Article:
    "Insert a new article into the database."
    session.add(article)
    session.commit()
    session.refresh(article)
    return article


In [ ]:
# | export
def get_article_by_id(session:Session,  # Active database session
                      article_id:int    # Article primary key
                     ) -> Article | None:
    "Get an article by its ID."
    return session.get(Article, article_id)


In [ ]:
# | export
def get_article_by_path(session:Session, # Active database session
                        file_path:str    # File path to match
                       ) -> Article | None:
    "Get an article by its file path."
    return session.exec(select(Article).where(Article.file_path == file_path)).first()


In [ ]:
# | export
def get_articles_by_website(session:Session, # Active database session
                            website_id:int   # Parent website ID
                           ) -> list[Article]:
    "Get all articles belonging to a website."
    return session.exec(select(Article).where(Article.website_id == website_id)).all()


In [ ]:
# | export
def delete_article(session:Session, # Active database session
                   article_id:int   # Article to delete
                  ) -> None:
    "Delete an article by ID. Raises ValueError if not found."
    article = session.get(Article, article_id)
    if article is None: raise ValueError(f"Article with id {article_id} not found")
    session.delete(article)
    session.commit()

In [ ]:
#| export
def update_article(session:Session,                          # Active database session
                   article_id:int,                           # Article to update
                   focus_keyword:str|None=None,              # New focus keyword
                   secondary_keywords:list[str]|None=None,   # New secondary keywords
                   target_goal:str|None=None,                # New target goal
                   url:str|None=None                         # New public URL
                  ) -> Article:
    "Update article fields and set last_optimized to now. Raises ValueError if not found."
    article = session.get(Article, article_id)
    if article is None: raise ValueError(f"Article with id {article_id} not found")
    if focus_keyword: article.focus_keyword = focus_keyword
    if secondary_keywords: article.secondary_keywords = secondary_keywords
    if target_goal: article.target_goal = target_goal
    if url: article.url = url
    article.last_optimized = datetime.now()
    session.add(article)
    session.commit()
    session.refresh(article)
    return article

In [ ]:

# | test
from fastcore.test import test_eq
from sqlmodel import create_engine, Session, SQLModel
from seo_rat.models import Website
from seo_rat.article import Article


In [ ]:
# | test
# create a test session

with get_session() as session:
    # insert
    article = insert_article(session, Article(
        website_id=1,
        file_path="/tmp/test.md",
        focus_keyword="seo",
        url="https://example.com/test"
    ))
    print("inserted:", article)

    # get by id
    print("by id:", get_article_by_id(session, article.id))

    # get by path
    print("by path:", get_article_by_path(session, "/tmp/test.md"))

    # get by website
    # print("by website:", get_articles_by_website(session, 1))

    # update
    updated = update_article(session, article.id, focus_keyword="seo tools", target_goal="rank #1")
    print("updated:", updated)

    # delete
    delete_article(session, article.id)
    print("deleted, verify gone:", get_article_by_id(session, article.id))


inserted: focus_keyword='seo' id=213 target_goal=None created_at=datetime.datetime(2026, 4, 4, 10, 39, 16, 204615) website_id=1 file_path='/tmp/test.md' url='https://example.com/test' secondary_keywords=None last_optimized=None
by id: focus_keyword='seo' id=213 target_goal=None created_at=datetime.datetime(2026, 4, 4, 10, 39, 16, 204615) website_id=1 file_path='/tmp/test.md' url='https://example.com/test' secondary_keywords=None last_optimized=None
by path: focus_keyword='seo' id=213 target_goal=None created_at=datetime.datetime(2026, 4, 4, 10, 39, 16, 204615) website_id=1 file_path='/tmp/test.md' url='https://example.com/test' secondary_keywords=None last_optimized=None
updated: focus_keyword='seo tools' id=213 target_goal='rank #1' created_at=datetime.datetime(2026, 4, 4, 10, 39, 16, 204615) website_id=1 file_path='/tmp/test.md' url='https://example.com/test' secondary_keywords=None last_optimized=datetime.datetime(2026, 4, 4, 10, 39, 16, 219366)
deleted, verify gone: None
